# Training the GST Regulatory LLM -- Local Workstation

Trains the 131.5M param model from this project's series on the extracted
GST regulatory corpus (`tharunpp007tk/gst-ruling-corpus` on Kaggle), from a
**manually downloaded copy on local disk**.

Before running: download and unzip the dataset from
https://www.kaggle.com/datasets/tharun007tk/gst-ruling-corpus/data, then
set `DATA_DIR` below to the folder you unzipped it into.

This is a genuinely separate notebook from the Colab/Kaggle ones -- it does
NOT branch on `google.colab` detection. That branching in the original
combined notebook only distinguished Colab from "everything else," and the
"everything else" branch hardcoded `/kaggle/working/...` paths that do not
exist, and that you likely don't have write permission to create, on a
normal workstation.


In [ ]:
!pip install -q torch numpy tiktoken pandas pyarrow

In [ ]:
# ---- Config -- EDIT THIS ----
MODEL_DIR = "../../04_model-architecture/01_main-chapter-code" # Local/Jupyter path
# MODEL_DIR = "/content/fingpt-131m-project/04_model-architecture/01_main-chapter-code" # Colab path
# MODEL_DIR = "/kaggle/working/fingpt-131m-project/04_model-architecture/01_main-chapter-code" # Kaggle path

DATA_DIR = "./data/gst-ruling-corpus"  # <-- folder you unzipped the Kaggle download into
CHECKPOINT_DIR = "./checkpoints"


In [ ]:
# ---- Config -- EDIT THIS ----
CONTEXT_LEN = 1024
BATCH_SIZE = 8
MAX_STEPS = 20000
EVAL_EVERY = 250
EVAL_ITERS = 50
LR = 3e-4
WEIGHT_DECAY = 0.1
PATIENCE = 10  # early-stopping: rounds with no val improvement before stopping


## Checkpoint storage

Local disk persists across runs by default (unlike Colab), so there's no
special mount step here -- just make sure `CHECKPOINT_DIR` is somewhere
that won't get wiped (not `/tmp`).

In [ ]:
import os

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints -> {os.path.abspath(CHECKPOINT_DIR)}")


## Load corpus from local disk

Looks for CSV or Parquet files under `DATA_DIR`. Tries to find files with
"train" and "val"/"valid" in the name first (matching a pre-made split);
falls back to a single file split 90/10 if it can't find two.

**Verify this actually matches your download** -- I don't have the real
file listing from Kaggle (the dataset page is JS-rendered and doesn't
expose it to a plain fetch), so confirm the glob below finds the right
files and that the text column really is named `text` before you trust the
tokenization step.

In [ ]:
import glob
import os
import pandas as pd

def _load_any(path):
    return pd.read_parquet(path) if path.endswith(".parquet") else pd.read_csv(path)

all_files = glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True) + \
            glob.glob(os.path.join(DATA_DIR, "**", "*.parquet"), recursive=True)
print("Found files:", all_files)

train_files = [f for f in all_files if "train" in os.path.basename(f).lower()]
val_files = [f for f in all_files if "val" in os.path.basename(f).lower()]

if train_files and val_files:
    train_df = _load_any(train_files[0])
    val_df = _load_any(val_files[0])
else:
    assert len(all_files) >= 1, f"No CSV/Parquet files found under {DATA_DIR} -- check DATA_DIR."
    full_df = _load_any(all_files[0]).sample(frac=1, random_state=42).reset_index(drop=True)
    split_idx = int(len(full_df) * 0.9)
    train_df, val_df = full_df.iloc[:split_idx], full_df.iloc[split_idx:]
    print(f"No separate train/val files found -- did a 90/10 split of {all_files[0]} instead.")

assert "text" in train_df.columns, f"Expected a 'text' column, got {list(train_df.columns)}"
print(f"Train rows: {len(train_df):,} | Val rows: {len(val_df):,}")
print("Sample:", train_df["text"].iloc[0][:200])


## Tokenize + pack

In [ ]:
import numpy as np
import tiktoken

enc = tiktoken.get_encoding("gpt2")
EOT = enc.eot_token

def tokenize_and_pack(texts, context_len):
    all_ids = []
    for t in texts:
        all_ids.extend(enc.encode(t))
        all_ids.append(EOT)
    arr = np.array(all_ids, dtype=np.uint16)
    n_seq = len(arr) // context_len
    return arr[: n_seq * context_len].reshape(n_seq, context_len)

train_data = tokenize_and_pack(train_df["text"].tolist(), CONTEXT_LEN)
val_data = tokenize_and_pack(val_df["text"].tolist(), CONTEXT_LEN)
print(f"Train sequences: {train_data.shape[0]:,}")
print(f"Val sequences:   {val_data.shape[0]:,}")


## Model architecture

In [ ]:
import sys
sys.path.insert(0, MODEL_DIR)
from model import GPTConfig, GPTModel


In [ ]:
cfg = GPTConfig(context_length=CONTEXT_LEN)
model = GPTModel(cfg)
print(f"Total parameters: {model.num_params():,}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
model = model.to(device)


## Training loop

In [ ]:
def get_batch(data, batch_size, device):
    idx = np.random.randint(0, data.shape[0], size=batch_size)
    seqs = torch.from_numpy(data[idx].astype(np.int64))
    x = seqs[:, :-1].contiguous()
    y = seqs[:, 1:].contiguous()
    return x.to(device), y.to(device)


def configure_optimizer(model, weight_decay, lr):
    decay_params, no_decay_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        (decay_params if param.dim() >= 2 else no_decay_params).append(param)
    return torch.optim.AdamW([
        {"params": decay_params, "weight_decay": weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ], lr=lr, betas=(0.9, 0.95))


@torch.no_grad()
def estimate_loss(model, data, batch_size, device, eval_iters=50):
    model.eval()
    losses = torch.zeros(eval_iters)
    for i in range(eval_iters):
        x, y = get_batch(data, batch_size, device)
        _, loss = model(x, y)
        losses[i] = loss.item()
    model.train()
    return losses.mean().item()


def save_checkpoint(path, model, optimizer, step, best_val_loss, no_improve_count, cfg):
    # no_improve_count is saved here -- the original version of this notebook
    # dropped it, which silently reset your early-stopping patience counter
    # to 0 on every resume. Fixed.
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "step": step,
        "best_val_loss": best_val_loss,
        "no_improve_count": no_improve_count,
        "torch_rng_state": torch.get_rng_state(),
        "numpy_rng_state": np.random.get_state(),
        "config": vars(cfg),
    }, path)


In [ ]:
# Resume support -- if a checkpoint already exists in CHECKPOINT_DIR (e.g.
# from a previous session that got cut off), pick up from there instead
# of starting over.
import os as _os

last_ckpt_path = _os.path.join(CHECKPOINT_DIR, "last.pt")
optimizer = configure_optimizer(model, WEIGHT_DECAY, LR)

start_step = 0
best_val_loss = float("inf")
no_improve_count = 0

if _os.path.exists(last_ckpt_path):
    print(f"Found existing checkpoint at {last_ckpt_path} -- resuming.")
    ckpt = torch.load(last_ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_step = ckpt["step"]
    best_val_loss = ckpt["best_val_loss"]
    no_improve_count = ckpt.get("no_improve_count", 0)  # .get() for backward compat
    if "torch_rng_state" in ckpt:
        torch.set_rng_state(ckpt["torch_rng_state"].to(torch.uint8).cpu())
    if "numpy_rng_state" in ckpt:
        np.random.set_state(ckpt["numpy_rng_state"])
    print(f"  Resumed at step {start_step}, best_val_loss so far: {best_val_loss:.4f}, "
          f"no_improve_count: {no_improve_count}")
else:
    print("No existing checkpoint -- starting fresh.")


In [ ]:
import time

model.train()
t0 = time.time()

for step in range(start_step, MAX_STEPS):
    x, y = get_batch(train_data, BATCH_SIZE, device)
    _, loss = model(x, y)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    if step % EVAL_EVERY == 0 or step == MAX_STEPS - 1:
        train_loss_est = estimate_loss(model, train_data, BATCH_SIZE, device, EVAL_ITERS)
        val_loss = estimate_loss(model, val_data, BATCH_SIZE, device, EVAL_ITERS)
        elapsed = time.time() - t0
        print(f"step {step:6d} | train_loss {train_loss_est:.4f} | "
              f"val_loss {val_loss:.4f} | {elapsed:.0f}s elapsed")

        save_checkpoint(last_ckpt_path, model, optimizer, step, best_val_loss,
                         no_improve_count, cfg)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve_count = 0
            save_checkpoint(_os.path.join(CHECKPOINT_DIR, "best.pt"),
                             model, optimizer, step, best_val_loss, no_improve_count, cfg)
            print(f"  New best val_loss: {best_val_loss:.4f} -- saved best.pt")
        else:
            no_improve_count += 1
            print(f"  No improvement ({no_improve_count}/{PATIENCE})")

        if no_improve_count >= PATIENCE:
            print(f"\nEarly stopping at step {step} -- best.pt is your model, not last.pt.")
            break

print("\nTraining complete (or interrupted -- re-run this cell to resume from last.pt).")
